# TikTok Profile Scraper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mamoo85/open-data-actor-notebooks/blob/main/notebooks/tiktok-public-profile-scraper.ipynb)

**What:** TikTok public profiles and latest posts: followers, likes, video count, bio, bio link, emails and links in bio, verified, business category, plus each recent video's plays, likes, comments, shares, saves, hashtags, music and engagement rate. Video URLs too. No login.

**Why:** run the [`tiktok-public-profile-scraper`](https://apify.com/transparent_meteorite/tiktok-public-profile-scraper) Apify actor from Python and analyse the result as a pandas DataFrame, with no scraping code to maintain.

> **Disclosure:** I am the author of this actor. It is pay-per-result; the free plan credit on a new Apify account covers a trial run like this one (this notebook uses a very small sample input).

**Privacy:** Public posts of public accounts only. Contact columns (bio emails, linked socials) are dropped before display.

## 1. Setup

In [1]:
!pip install -q apify-client pandas

import os, getpass
try:
    from google.colab import userdata
    APIFY_TOKEN = userdata.get('APIFY_TOKEN')          # Colab: add it under the key icon (Secrets)
except Exception:
    APIFY_TOKEN = os.environ.get('APIFY_TOKEN') or getpass.getpass('Apify API token: ')
assert APIFY_TOKEN, 'An Apify API token is required (a free account is enough).'
print('Token loaded.')

Token loaded.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Input
A deliberately small sample so the run is cheap and quick. See the [actor page](https://apify.com/transparent_meteorite/tiktok-public-profile-scraper) for every option.

In [2]:
run_input = {'profiles': ['natgeo'], 'resultsPerPage': 3, 'resultsType': 'posts', 'maxItems': 3}

## 3. Run the actor and read the dataset

In [3]:
from apify_client import ApifyClient

client = ApifyClient(APIFY_TOKEN)
run = client.actor('transparent_meteorite/tiktok-public-profile-scraper').call(run_input=run_input)
# apify-client v2 returns a dict, v3 a Run object; handle both
get = (lambda k, a: run[k]) if isinstance(run, dict) else (lambda k, a: getattr(run, a))
items = list(client.dataset(get('defaultDatasetId', 'default_dataset_id')).iterate_items())
print('Run status:', get('status', 'status'), '| items:', len(items))

[apify.tiktok-public-profile-scraper runId:qWR0PWxCv4m2ajKoq] -> Status: RUNNING, Message: 


[apify.tiktok-public-profile-scraper runId:qWR0PWxCv4m2ajKoq] -> 2026-10-14T07:15:45.614Z ACTOR: Pulling container image of build KQPmUD1r2jecR6Qvb from registry.


[apify.tiktok-public-profile-scraper runId:qWR0PWxCv4m2ajKoq] -> 2026-10-14T07:15:46.396Z ACTOR: Creating container.


[apify.tiktok-public-profile-scraper runId:qWR0PWxCv4m2ajKoq] -> 2026-10-14T07:15:46.459Z ACTOR: Starting container.


[apify.tiktok-public-profile-scraper runId:qWR0PWxCv4m2ajKoq] -> 2026-10-14T07:15:46.459Z ACTOR: Running under "LIMITED_PERMISSIONS".


...

[apify.tiktok-public-profile-scraper runId:qWR0PWxCv4m2ajKoq] -> 2026-10-14T07:16:00.350Z Done. Delivered 3 rows (charged 3); profiles ok 1, not found 0, failed 0; filtered 0; requests 8, ok 6.


[apify.tiktok-public-profile-scraper runId:qWR0PWxCv4m2ajKoq] -> Status: RUNNING, Message: Done: 3 items


[apify.tiktok-public-profile-scraper runId:qWR0PWxCv4m2ajKoq] -> 2026-10-14T07:16:00.449Z INFO  [Status message]: Done: 3 items


[apify.tiktok-public-profile-scraper runId:qWR0PWxCv4m2ajKoq] -> 2026-10-14T07:16:00.755Z Thanks for using TikTok Public Profile Scraper. Found a bug or need a field? Open an issue on the Issues tab (https://console.apify.com/actors/8O6RGrfKC0dOdpACm/issues) with the run URL and we will reply there.


Run status: SUCCEEDED | items: 3


## 4. Tabulate

Public posts of public accounts only. Contact columns (bio emails, linked socials) are dropped before display.

In [4]:
import pandas as pd
pd.set_option('display.max_colwidth', 70)
pd.set_option('display.width', 200)

df = pd.DataFrame(items)
df['account'] = df['authorMeta'].apply(lambda a: a.get('name') if isinstance(a, dict) else None)

# Privacy: remove contact / personal-data columns before anything is displayed
DROP = ["email", "phone", "author", "registeredagent", "contractor", "officer", "linkedin", "facebook", "instagram", "youtube", "tiktok", "socials", "twitter"]
KEEP = ["emailCount", "phoneCount", "emailProvider"]
df = df[[c for c in df.columns if c in KEEP or not any(p in c.lower() for p in DROP)]]
SHOW = [c for c in ["account", "createTimeISO", "playCount", "diggCount", "commentCount", "shareCount", "engagementRate", "isSponsored", "text"] if c in df.columns]
df[SHOW]

,account,createTimeISO,playCount,diggCount,commentCount,shareCount,engagementRate,isSponsored,text
0,natgeo,2026-09-10T13:04:07.000Z,163300,22500,346,742,14.44,False,Presented by @ROLEX. Welcome to Africa 🌍 Africa Earth's Wild Home ...
1,natgeo,2026-10-13T22:12:45.000Z,6897,516,12,4,7.71,False,"For several weeks each fall and winter, schools of migrating mulle..."
2,natgeo,2026-10-12T16:02:43.000Z,19200,1680,30,87,9.36,False,Presented by @ROLEX. Pangolins are notoriously difficult to rehabi...


## 5. A quick analysis

In [5]:
for c in ['playCount','diggCount','commentCount','shareCount']:
    df[c]=pd.to_numeric(df[c],errors='coerce')
print(df[['playCount','diggCount','commentCount','shareCount']].describe().loc[['count','mean','max']])
df.sort_values('playCount',ascending=False)[['account','playCount','diggCount']].head(5)

           playCount  diggCount  commentCount  shareCount
count       3.000000        3.0      3.000000    3.000000
mean    63132.333333     8232.0    129.333333  277.666667
max    163300.000000    22500.0    346.000000  742.000000


,account,playCount,diggCount
0,natgeo,163300,22500
2,natgeo,19200,1680
1,natgeo,6897,516


## Next steps

- Raise the limits in the input cell to run a larger sample.
- Schedule the actor or call it from your own pipeline: [tiktok-public-profile-scraper on the Apify Store](https://apify.com/transparent_meteorite/tiktok-public-profile-scraper).
- Check the Store page for the current pay-per-result price before scaling up.

Generated by `tools/generate.py`.